In [1]:
# related mostly to https://www.kaggle.com/phuongpm/mbti-prediction/data

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import sklearn

import re
import string

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier, plot_importance
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.feature_selection import SelectFromModel
from itertools import compress

from pylab import rcParams
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

from nltk.corpus import stopwords 
from nltk import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [3]:
data = pd.read_csv('input/kaggle_mbti.csv')
data.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


In [4]:
dist = data['type'].value_counts()
dist

INFP    1832
INFJ    1470
INTP    1304
INTJ    1091
ENTP     685
ENFP     675
ISTP     337
ISFP     271
ENTJ     231
ISTJ     205
ENFJ     190
ISFJ     166
ESTP      89
ESFP      48
ESFJ      42
ESTJ      39
Name: type, dtype: int64

In [5]:
dist.index

Index(['INFP', 'INFJ', 'INTP', 'INTJ', 'ENTP', 'ENFP', 'ISTP', 'ISFP', 'ENTJ',
       'ISTJ', 'ENFJ', 'ISFJ', 'ESTP', 'ESFP', 'ESFJ', 'ESTJ'],
      dtype='object')

In [6]:
data['seperated_post'] = data['posts'].apply(lambda x: x.strip().split("|||"))
data['num_post'] = data['seperated_post'].apply(lambda x: len(x))
data.head()

,type,posts,seperated_post,num_post
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,"['http://www.youtube.com/watch?v=qsXHcwe3krw, ...",50
1,ENTP,'I'm finding the lack of me in these posts ver...,['I'm finding the lack of me in these posts ve...,50
2,INTP,'Good one _____ https://www.youtube.com/wat...,['Good one _____ https://www.youtube.com/wa...,50
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","['Dear INTP, I enjoyed our conversation the ...",50
4,ENTJ,'You're fired.|||That's another silly misconce...,"['You're fired., That's another silly misconce...",50


In [7]:
# Before expanding the dataframe, give everyone an unique ID?
data['id'] = data.index
data.head()

,type,posts,seperated_post,num_post,id
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,"['http://www.youtube.com/watch?v=qsXHcwe3krw, ...",50,0
1,ENTP,'I'm finding the lack of me in these posts ver...,['I'm finding the lack of me in these posts ve...,50,1
2,INTP,'Good one _____ https://www.youtube.com/wat...,['Good one _____ https://www.youtube.com/wa...,50,2
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","['Dear INTP, I enjoyed our conversation the ...",50,3
4,ENTJ,'You're fired.|||That's another silly misconce...,"['You're fired., That's another silly misconce...",50,4


In [8]:
expanded_df = pd.DataFrame(data['seperated_post'].tolist(), index=data['id']).stack().reset_index(level=1, drop=True).reset_index(name='idposts')
expanded_df.head()

,id,idposts
0,0,'http://www.youtube.com/watch?v=qsXHcwe3krw
1,0,http://41.media.tumblr.com/tumblr_lfouy03PMA1q...
2,0,enfp and intj moments https://www.youtube.com...
3,0,What has been the most life-changing experienc...
4,0,http://www.youtube.com/watch?v=vXZeYwwRDw8 h...


In [9]:
expanded_df=expanded_df.join(data.set_index('id'), on='id', how = 'left')
expanded_df=expanded_df.drop(columns=['posts','seperated_post','num_post'])
expanded_df.head()

,id,idposts,type
0,0,'http://www.youtube.com/watch?v=qsXHcwe3krw,INFJ
1,0,http://41.media.tumblr.com/tumblr_lfouy03PMA1q...,INFJ
2,0,enfp and intj moments https://www.youtube.com...,INFJ
3,0,What has been the most life-changing experienc...,INFJ
4,0,http://www.youtube.com/watch?v=vXZeYwwRDw8 h...,INFJ


In [ ]:
def clean_text(text):
    # Lemmatizer | Stemmatizer
    stemmer = PorterStemmer()

    # Cache the stop words for speed 
    cachedStopWords = stopwords.words("english")

    text = re.sub(r'http[^\s]*', '',text)
    text = re.sub('[0-9]+','', text).lower()
    text = re.sub('@[a-z0-9]+', 'user', text)
    text = re.sub("[^a-zA-Z]", " ", text) #keep only words

    # some additional info about this that https://towardsdatascience.com/fuzzy-string-match-with-python-on-large-dataset-and-why-you-should-not-use-fuzzywuzzy-4ec9f0defcd
    stemmer.stem(" ".join([w for w in text.split(' ') if w not in cachedStopWords]))

    text = re.sub(' +', ' ', text).lower() # Remove spaces > 1
    text = re.sub('[%s]*' % string.punctuation, '',text)

    return text

In [11]:
final_df = expanded_df.copy()

In [12]:
final_df['idposts'] = final_df['idposts'].apply(clean_text)

In [13]:
final_df.head()

,id,idposts,type
0,0,,INFJ
1,0,,INFJ
2,0,enfp and intj moments sportscenter not top ten...,INFJ
3,0,what has been the most life changing experienc...,INFJ
4,0,on repeat for most of today,INFJ


In [14]:
cleaned_df = final_df.groupby('id')['idposts'].apply(list).reset_index()
cleaned_df.head()

,id,idposts
0,0,"[ , , enfp and intj moments sportscenter not t..."
1,1,[ i m finding the lack of me in these posts ve...
2,2,"[ good one , of course to which i say i know t..."
3,3,[ dear intp i enjoyed our conversation the oth...
4,4,"[ you re fired , that s another silly misconce..."


In [15]:
data['clean_posts'] = cleaned_df['idposts'].apply(lambda x: ' '.join(x))
data.head()

,type,posts,seperated_post,num_post,id,clean_posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,"['http://www.youtube.com/watch?v=qsXHcwe3krw, ...",50,0,enfp and intj moments sportscenter not top ...
1,ENTP,'I'm finding the lack of me in these posts ver...,['I'm finding the lack of me in these posts ve...,50,1,i m finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...,['Good one _____ https://www.youtube.com/wa...,50,2,good one of course to which i say i know tha...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","['Dear INTP, I enjoyed our conversation the ...",50,3,dear intp i enjoyed our conversation the othe...
4,ENTJ,'You're fired.|||That's another silly misconce...,"['You're fired., That's another silly misconce...",50,4,you re fired that s another silly misconcept...


In [16]:
def clean_text_extra_spaces(text):
    stemmer = PorterStemmer()
    
    text = re.sub(' +', ' ', text).lower() # Remove spaces > 1

    return re.sub('[%s]*' % string.punctuation, '',text)

In [17]:
data['clean_posts'] = data['clean_posts'].apply(clean_text_extra_spaces)
data.head()

,type,posts,seperated_post,num_post,id,clean_posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,"['http://www.youtube.com/watch?v=qsXHcwe3krw, ...",50,0,enfp and intj moments sportscenter not top te...
1,ENTP,'I'm finding the lack of me in these posts ver...,['I'm finding the lack of me in these posts ve...,50,1,i m finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...,['Good one _____ https://www.youtube.com/wa...,50,2,good one of course to which i say i know that...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","['Dear INTP, I enjoyed our conversation the ...",50,3,dear intp i enjoyed our conversation the othe...
4,ENTJ,'You're fired.|||That's another silly misconce...,"['You're fired., That's another silly misconce...",50,4,you re fired that s another silly misconcepti...


In [18]:
data['E_I'] = data['type'].apply(lambda x: 1 if x[0] == 'E' else 0)
data['S_N'] = data['type'].apply(lambda x: 1 if x[1] == 'S' else 0)
data['T_F'] = data['type'].apply(lambda x: 1 if x[2] == 'T' else 0)
data['J_P'] = data['type'].apply(lambda x: 1 if x[3] == 'J' else 0)

In [19]:
data.head()

,type,posts,seperated_post,num_post,id,clean_posts,E_I,S_N,T_F,J_P
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,"['http://www.youtube.com/watch?v=qsXHcwe3krw, ...",50,0,enfp and intj moments sportscenter not top te...,0,0,0,1
1,ENTP,'I'm finding the lack of me in these posts ver...,['I'm finding the lack of me in these posts ve...,50,1,i m finding the lack of me in these posts ver...,1,0,1,0
2,INTP,'Good one _____ https://www.youtube.com/wat...,['Good one _____ https://www.youtube.com/wa...,50,2,good one of course to which i say i know that...,0,0,1,0
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","['Dear INTP, I enjoyed our conversation the ...",50,3,dear intp i enjoyed our conversation the othe...,0,0,1,1
4,ENTJ,'You're fired.|||That's another silly misconce...,"['You're fired., That's another silly misconce...",50,4,you re fired that s another silly misconcepti...,1,0,1,1


In [20]:
data=data.drop(columns=['posts','seperated_post','num_post','id'])

In [21]:
data.head()

,type,clean_posts,E_I,S_N,T_F,J_P
0,INFJ,enfp and intj moments sportscenter not top te...,0,0,0,1
1,ENTP,i m finding the lack of me in these posts ver...,1,0,1,0
2,INTP,good one of course to which i say i know that...,0,0,1,0
3,INTJ,dear intp i enjoyed our conversation the othe...,0,0,1,1
4,ENTJ,you re fired that s another silly misconcepti...,1,0,1,1


In [ ]:
data.to_csv('data/data_clean.csv' ,index=False)